# Assignment 3: Image Classification using CNN (CIFAR-10)
**Aim:** Build an image classification model using a CNN in 4 stages:

a. Loading and preprocessing the image data  b. Defining the model architecture  c. Training the model  d. Estimating performance

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

I0000 00:00:1793641313.935386   42749 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1793641315.726617   42749 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## a. Loading and preprocessing the image data

In [2]:
# Path of dataset folder (change if your folder is somewhere else)
train_dir = "dataset/cifar-10-img/train"
test_dir  = "dataset/cifar-10-img/test"

datagen = ImageDataGenerator(rescale=1.0/255)   # pixel values 0-255 -> 0-1

train_gen = datagen.flow_from_directory(train_dir, target_size=(32, 32),
                                        color_mode="rgb", batch_size=50000,
                                        class_mode="categorical", shuffle=True)
test_gen  = datagen.flow_from_directory(test_dir, target_size=(32, 32),
                                        color_mode="rgb", batch_size=10000,
                                        class_mode="categorical", shuffle=True)

# One big batch = whole dataset loaded as numpy arrays
x_train, y_train = train_gen[0]
x_test,  y_test  = test_gen[0]
class_names = list(train_gen.class_indices.keys())

print("x_train:", x_train.shape, " y_train:", y_train.shape)
print("x_test :", x_test.shape,  " y_test :", y_test.shape)
print("Classes:", class_names)

Found 40079 images belonging to 10 classes.


Found 9921 images belonging to 10 classes.


x_train: (40079, 32, 32, 3)  y_train: (40079, 10)
x_test : (9921, 32, 32, 3)  y_test : (9921, 10)
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [3]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[np.argmax(y_train[i])])
    plt.axis("off")
plt.show()

/tmp/ipykernel_42749/676615008.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## b. Defining the model's architecture
Conv2D (feature extraction with filters) → MaxPooling (downsampling) → Flatten → Dense → Softmax output (10 classes)

In [4]:
model = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax"),
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,048,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,116,970 (4.26 MB)

 Trainable params: 1,116,970 (4.26 MB)

 Non-trainable params: 0 (0.00 B)

## c. Training the model

In [5]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

history = model.fit(x_train, y_train,
                    validation_data=(x_test, y_test),
                    epochs=15, batch_size=64)

Epoch 1/15


  1/627 ━━━━━━━━━━━━━━━━━━━━ 8:50 847ms/step - accuracy: 0.0781 - loss: 2.3174

  2/627 ━━━━━━━━━━━━━━━━━━━━ 51s 82ms/step - accuracy: 0.0938 - loss: 2.3209  

  3/627 ━━━━━━━━━━━━━━━━━━━━ 51s 83ms/step - accuracy: 0.0938 - loss: 2.3239

  4/627 ━━━━━━━━━━━━━━━━━━━━ 52s 84ms/step - accuracy: 0.1055 - loss: 2.3198

  5/627 ━━━━━━━━━━━━━━━━━━━━ 53s 85ms/step - accuracy: 0.1063 - loss: 2.3180

... (9,411 outputs left out)

624/627 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.8039 - loss: 0.5492

625/627 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.8039 - loss: 0.5493

626/627 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.8039 - loss: 0.5492

627/627 ━━━━━━━━━━━━━━━━━━━━ 56s 89ms/step - accuracy: 0.8040 - loss: 0.5491 - val_accuracy: 0.7755 - val_loss: 0.6658


## d. Estimating the model's performance

In [6]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc*100:.2f}%")

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d",
            xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.show()

Test loss: 0.6658
Test accuracy: 77.55%


              precision    recall  f1-score   support

    airplane       0.79      0.83      0.81       986
  automobile       0.91      0.88      0.89      1057
        bird       0.68      0.65      0.67       982
         cat       0.58      0.59      0.59       974
        deer       0.77      0.73      0.75      1001
         dog       0.71      0.63      0.67       975
        frog       0.80      0.82      0.81      1013
       horse       0.77      0.87      0.82       975
        ship       0.86      0.88      0.87       955
       truck       0.86      0.88      0.87      1003

    accuracy                           0.78      9921
   macro avg       0.77      0.77      0.77      9921
weighted avg       0.77      0.78      0.77      9921



/tmp/ipykernel_42749/4274663811.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Loss"); plt.xlabel("Epoch"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Accuracy"); plt.xlabel("Epoch"); plt.legend()
plt.show()

/tmp/ipykernel_42749/1834603888.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
n = 25   # change index to test another image
plt.imshow(x_test[n]); plt.axis("off"); plt.show()
pred = model.predict(x_test[n:n+1], verbose=0)
print("Predicted:", class_names[np.argmax(pred)])
print("Actual   :", class_names[np.argmax(y_test[n])])

Predicted: ship
Actual   : deer


/tmp/ipykernel_42749/714987791.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(x_test[n]); plt.axis("off"); plt.show()
